# 🧠 Brain Tumor Detection — Training Notebook
**Model:** EfficientNet-B0 | **Framework:** PyTorch

---
### Setup Instructions
1. Upload your `brainMRI.zip` directly into the Colab file explorer.
2. Run `!unzip -q /content/brainMRI.zip -d /content/` in a new cell to extract the data.
3. Ensure the `Training` and `Testing` folders are extracted directly under `/content/`.
4. Change runtime type to **GPU (T4)**.
5. Run all cells.

In [ ]:
!pip install -q timm scikit-learn seaborn scipy sympy --upgrade

In [ ]:
!unzip -q /content/brainMRI.zip -d /content/

In [ ]:
import sympy
import sympy.core
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from scipy.fft import fft2, ifft2, fftshift, ifftshift
import time, copy

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
CONFIG = {
    'data_dir': '/content',
    'img_size': 224,
    'batch_size': 32,
    'num_classes': 4,
    'epochs': 15,
    'lr': 1e-4,
    'weight_decay': 1e-3,
    'model_name': 'efficientnet_b0',
    'save_path': 'brain_tumor_efficientnet_b0.pth'
}

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop((CONFIG['img_size'], CONFIG['img_size']), scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dir = os.path.join(CONFIG['data_dir'], 'Training')
test_dir = os.path.join(CONFIG['data_dir'], 'Testing')

base_train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
base_val_dataset = datasets.ImageFolder(train_dir, transform=val_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=val_transforms)

num_train_samples = len(base_train_dataset)
indices = torch.randperm(num_train_samples).tolist()
train_size = int(0.85 * num_train_samples)

train_idx = indices[:train_size]
val_idx = indices[train_size:]

train_dataset = Subset(base_train_dataset, train_idx)
val_dataset = Subset(base_val_dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)} | Test samples: {len(test_dataset)}")
print(f"Classes: {base_train_dataset.classes}")

In [ ]:
def plot_pca(loader, num_samples=500):
    features, labels = [], []
    for imgs, lbls in loader:
        features.append(imgs.view(imgs.size(0), -1).numpy())
        labels.append(lbls.numpy())
        if len(features) * CONFIG['batch_size'] >= num_samples: break

    X = np.vstack(features)
    y = np.concatenate(labels)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.6)
    plt.legend(handles=scatter.legend_elements()[0], labels=base_train_dataset.classes)
    plt.title('PCA of Raw Image Features')
    plt.show()

plot_pca(train_loader)

In [ ]:
def ifft_enhance_demo(img_tensor):
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
    h, w, c = img_np.shape
    enhanced = np.zeros_like(img_np)

    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((Y - cy)**2 + (X - cx)**2)
    mask = 1.0 + 0.35 * (1 - np.exp(-(dist**2)/(2*(min(h, w)*0.25)**2)))

    for ch in range(c):
        spectrum = fftshift(fft2(img_np[:, :, ch]))
        restored = np.real(ifft2(ifftshift(spectrum * mask)))
        enhanced[:, :, ch] = np.clip(restored, 0, 1)

    return img_np, enhanced

sample_img = next(iter(train_loader))[0][0]
orig, enh = ifft_enhance_demo(sample_img)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(orig); ax1.set_title('Original')
ax2.imshow(enh); ax2.set_title('IFFT Enhanced')
plt.show()

In [ ]:
model = timm.create_model(CONFIG['model_name'], pretrained=True, num_classes=CONFIG['num_classes'], drop_rate=0.4).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_acc = 0.0
best_wts = copy.deepcopy(model.state_dict())

for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss, train_correct = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()

    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()

    scheduler.step()

    t_loss = train_loss / len(train_dataset)
    t_acc = train_correct / len(train_dataset)
    v_loss = val_loss / len(val_dataset)
    v_acc = val_correct / len(val_dataset)

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    if v_acc > best_acc:
        best_acc = v_acc
        best_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch+1:02d} | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")

model.load_state_dict(best_wts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_title('Accuracy')
axes[1].legend()
plt.show()

In [ ]:
model.eval()
test_loss, test_correct = 0.0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        test_correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Test Loss: {test_loss/len(test_dataset):.4f} | Test Acc: {test_correct/len(test_dataset):.4f}")

In [ ]:
print(classification_report(all_labels, all_preds, target_names=base_train_dataset.classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=base_train_dataset.classes, yticklabels=base_train_dataset.classes)
plt.title('Test Confusion Matrix')
plt.show()

In [ ]:
save_dict = {
    'model_name': CONFIG['model_name'],
    'num_classes': CONFIG['num_classes'],
    'class_names': base_train_dataset.classes,
    'state_dict': model.state_dict(),
    'best_val_acc': best_acc,
    'img_size': CONFIG['img_size']
}
torch.save(save_dict, CONFIG['save_path'])

from google.colab import files
files.download(CONFIG['save_path'])